# Market Maker Simulator — Interactive Walkthrough

This notebook explores three market-making strategies through simulation:

1. **Naive (Fixed Spread)** — quotes a constant spread, ignores all risk
2. **Inventory-Based** — skews quotes to manage inventory position  
3. **Avellaneda–Stoikov (2008)** — optimal quoting under CARA utility

We'll run single simulations to build intuition, then use Monte Carlo analysis to compare strategy performance statistically.

In [ ]:
import sys
sys.path.insert(0, "..")

from market_maker import (
    NaiveStrategy, InventoryStrategy, AvellanedaStoikov,
    PriceProcess, TraderConfig, SimulationConfig,
    run_simulation, monte_carlo,
)
from market_maker.visualization import (
    plot_simulation,
    plot_strategy_comparison,
    plot_single_path_comparison,
)
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Setup

Configure the simulation environment:
- **Asset**: starts at 100, no drift, volatility σ = 2
- **Traders**: arrive at rate λ = 100/session, 30% are informed
- **Time**: 10,000 steps over one trading session

In [ ]:
price = PriceProcess(s0=100.0, mu=0.0, sigma=2.0)
traders = TraderConfig(arrival_rate=100.0, informed_fraction=0.3, noise_tolerance=1.0)

config = SimulationConfig(
    T=1.0,
    n_steps=10_000,
    price_process=price,
    trader_config=traders,
    trade_impact=0.05,
    info_leakage=0.005,
    seed=42,
)

print(f"Time horizon: {config.T}")
print(f"Time steps:   {config.n_steps}")
print(f"dt:           {config.T / config.n_steps:.6f}")
print(f"Volatility:   {price.sigma}")
print(f"Arrival rate: {traders.arrival_rate}")
print(f"Informed %:   {traders.informed_fraction:.0%}")

## 2. Single Simulation — Naive Strategy

The naive strategy quotes a fixed spread around its mid-price estimate:

$$\text{bid} = m - \delta/2, \qquad \text{ask} = m + \delta/2$$

It makes no attempt to manage inventory, leaving it exposed to both inventory risk and adverse selection.

In [ ]:
naive = NaiveStrategy(half_spread=0.5)
result_naive = run_simulation(naive, config)

print(f"Strategy:     {result_naive.strategy_name}")
print(f"Final P&L:    {result_naive.final_pnl:+.2f}")
print(f"Total trades: {result_naive.n_trades} (buys: {result_naive.n_buys}, sells: {result_naive.n_sells})")
print(f"Max |inv|:    {result_naive.max_inventory}")

fig = plot_simulation(result_naive)
plt.show()

## 3. Single Simulation — Inventory-Based Strategy

Skews quotes proportionally to inventory:

$$\text{bid} = m - \delta/2 - \kappa q, \qquad \text{ask} = m + \delta/2 - \kappa q$$

When long ($q > 0$), both quotes shift down → encourages others to buy from us → reduces inventory.

In [ ]:
inventory_strat = InventoryStrategy(half_spread=0.5, skew=0.1)
result_inv = run_simulation(inventory_strat, config)

print(f"Strategy:     {result_inv.strategy_name}")
print(f"Final P&L:    {result_inv.final_pnl:+.2f}")
print(f"Total trades: {result_inv.n_trades} (buys: {result_inv.n_buys}, sells: {result_inv.n_sells})")
print(f"Max |inv|:    {result_inv.max_inventory}")

fig = plot_simulation(result_inv)
plt.show()

## 4. Single Simulation — Avellaneda–Stoikov

The optimal strategy under CARA utility. Two key equations:

**Reservation price** (inventory-penalized fair value):
$$r = s - q \cdot \gamma \cdot \sigma^2 \cdot (T - t)$$

**Optimal spread**:
$$\delta^* = \gamma \cdot \sigma^2 \cdot (T - t) + \frac{2}{\gamma} \ln\left(1 + \frac{\gamma}{\kappa}\right)$$

Notice how the spread *narrows* as $T - t \to 0$ — the MM becomes more aggressive to unwind inventory before the session ends.

In [ ]:
as_strat = AvellanedaStoikov(gamma=0.1, kappa=1.5)
result_as = run_simulation(as_strat, config)

print(f"Strategy:     {result_as.strategy_name}")
print(f"Final P&L:    {result_as.final_pnl:+.2f}")
print(f"Total trades: {result_as.n_trades} (buys: {result_as.n_buys}, sells: {result_as.n_sells})")
print(f"Max |inv|:    {result_as.max_inventory}")

fig = plot_simulation(result_as)
plt.show()

## 5. Head-to-Head Comparison (Same Price Path)

All three strategies face the **same price path and trader arrivals**. This isolates the impact of the quoting strategy on P&L and inventory.

In [ ]:
comparison = {
    naive.name: result_naive,
    inventory_strat.name: result_inv,
    as_strat.name: result_as,
}

fig = plot_single_path_comparison(comparison)
plt.show()

# Summary table
print(f"{'Strategy':<25} {'P&L':>8} {'Trades':>8} {'Max|Inv|':>10} {'Sharpe':>8}")
print("-" * 65)
for name, r in comparison.items():
    print(f"{name:<25} {r.final_pnl:>+8.2f} {r.n_trades:>8d} {r.max_inventory:>10d} {r.pnl_sharpe:>8.2f}")

## 6. Monte Carlo Analysis

A single simulation can be misleading — the price path might favor one strategy by luck. We run **500 independent simulations** per strategy to compute robust statistics.

This takes about 30–60 seconds.

In [ ]:
N_SIMS = 500

mc_results = {}
for name, strategy in [("Naive (Fixed Spread)", naive), 
                        ("Inventory-Based", inventory_strat),
                        ("Avellaneda-Stoikov", as_strat)]:
    print(f"Running {name}...", end=" ", flush=True)
    mc_results[name] = monte_carlo(strategy, config, n_simulations=N_SIMS)
    pnls = [r.final_pnl for r in mc_results[name]]
    print(f"E[P&L] = {np.mean(pnls):+.2f}, σ = {np.std(pnls):.2f}")

fig = plot_strategy_comparison(mc_results)
plt.show()

## 7. Sensitivity Analysis — Risk Aversion (γ)

How does the Avellaneda–Stoikov strategy's risk aversion parameter $\gamma$ affect performance? Higher $\gamma$ means:
- Wider spreads (more conservative)
- More aggressive inventory control
- Fewer trades, but each trade is safer

In [ ]:
gammas = [0.01, 0.05, 0.1, 0.5, 1.0]
n_sims_sensitivity = 200

gamma_results = {}
for gamma in gammas:
    strat = AvellanedaStoikov(gamma=gamma, kappa=1.5)
    runs = monte_carlo(strat, config, n_simulations=n_sims_sensitivity)
    pnls = [r.final_pnl for r in runs]
    max_invs = [r.max_inventory for r in runs]
    gamma_results[gamma] = {
        "mean_pnl": np.mean(pnls),
        "std_pnl": np.std(pnls),
        "sharpe": np.mean(pnls) / np.std(pnls) if np.std(pnls) > 0 else 0,
        "mean_max_inv": np.mean(max_invs),
    }
    print(f"γ = {gamma:<5.2f}  E[P&L] = {np.mean(pnls):+7.2f}  σ = {np.std(pnls):6.2f}  "
          f"Sharpe = {gamma_results[gamma]['sharpe']:5.2f}  E[Max|Inv|] = {np.mean(max_invs):.1f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

g = list(gamma_results.keys())
ax1.plot(g, [gamma_results[x]["mean_pnl"] for x in g], "o-", color="#2ecc71", linewidth=2)
ax1.fill_between(g,
    [gamma_results[x]["mean_pnl"] - gamma_results[x]["std_pnl"] for x in g],
    [gamma_results[x]["mean_pnl"] + gamma_results[x]["std_pnl"] for x in g],
    alpha=0.2, color="#2ecc71")
ax1.set_xlabel("Risk Aversion (γ)")
ax1.set_ylabel("P&L")
ax1.set_title("Expected P&L vs. Risk Aversion", fontweight="bold")
ax1.set_xscale("log")

ax2.plot(g, [gamma_results[x]["mean_max_inv"] for x in g], "o-", color="#8e44ad", linewidth=2)
ax2.set_xlabel("Risk Aversion (γ)")
ax2.set_ylabel("Avg Max |Inventory|")
ax2.set_title("Inventory Exposure vs. Risk Aversion", fontweight="bold")
ax2.set_xscale("log")

plt.tight_layout()
plt.show()

## Conclusions

1. The **naive strategy** is a losing proposition under adverse selection — informed traders systematically exploit stale quotes.
2. **Inventory-based skewing** significantly reduces risk but is a heuristic — the skew parameter must be tuned.
3. The **Avellaneda–Stoikov** model provides a principled, utility-theoretic approach that jointly optimizes the spread *and* inventory management, achieving the best risk-adjusted returns.
4. The **risk aversion parameter** (γ) controls the tradeoff between profitability and safety — higher γ reduces variance at the cost of expected P&L.

---

*Reference: Avellaneda, M. & Stoikov, S. (2008). "High-frequency trading in a limit order book." Quantitative Finance, 8(3), 217–224.*